### **Requirements**

In [ ]:
pip install pycaret ydata_profiling

---

# **Regression**

## **Libraries**

In [ ]:
import pandas as pd
from ydata_profiling import ProfileReport
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from pycaret.regression import setup, compare_models, plot_model
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.impute import KNNImputer, SimpleImputer
from sklearn.ensemble import IsolationForest
from catboost import CatBoostRegressor
import optuna
import numpy as np
from sklearn.metrics import root_mean_squared_log_error
from sklearn.metrics import mean_squared_log_error
import numpy as np
import time
from catboost import CatBoostRegressor, Pool
from sklearn.model_selection import KFold
import lightgbm as lgb

warnings.filterwarnings("ignore", category=RuntimeWarning)

---

## **Dataset**

**File Descriptions and Data Field Information**

**train.csv**

* The training data, comprising time series of features store_nbr, family, and onpromotion as well as the target sales.
* store_nbr identifies the store at which the products are sold.
* family identifies the type of product sold.
* sales gives the total sales for a product family at a particular store at a given date. Fractional values are possible since products can be sold in fractional units (1.5 kg of cheese, for instance, as opposed to 1 bag of chips).
* onpromotion gives the total number of items in a product family that were being promoted at a store at a given date.

**test.csv**

* The test data, having the same features as the training data. You will predict the target sales for the dates in this file.
* The dates in the test data are for the 15 days after the last date in the training data.

**sample_submission.csv**

* A sample submission file in the correct format.
stores.csv
* Store metadata, including city, state, type, and cluster.
cluster is a grouping of similar stores.

**oil.csv**

* Daily oil price. Includes values during both the train and test data timeframes. (Ecuador is an oil-dependent country and it's economical health is highly vulnerable to shocks in oil prices.)

**holidays_events.csv**

* Holidays and Events, with metadata
* NOTE: Pay special attention to the transferred column. A holiday that is transferred officially falls on that calendar day, but was moved to another date by the government. A transferred day is more like a normal day than a holiday. To find the day that it was actually celebrated, look for the corresponding row where type is Transfer. For example, the holiday Independencia de Guayaquil was transferred from 2012-10-09 to 2012-10-12, which means it was celebrated on 2012-10-12. Days that are type Bridge are extra days that are added to a holiday (e.g., to extend the break across a long weekend). These are frequently made up by the type Work Day which is a day not normally scheduled for work (e.g., Saturday) that is meant to payback the Bridge.
* Additional holidays are days added a regular calendar holiday, for example, as typically happens around Christmas (making Christmas Eve a holiday).

**Additional Notes**

* Wages in the public sector are paid every two weeks on the 15 th and on the last day of the month. Supermarket sales could be affected by this.
* A magnitude 7.8 earthquake struck Ecuador on April 16, 2016. People rallied in relief efforts donating water and other first need products which greatly affected supermarket sales for several weeks after the earthquake.

In [ ]:
holidays_events = pd.read_csv('/kaggle/input/store-sales-time-series-forecasting/holidays_events.csv')
oil = pd.read_csv('/kaggle/input/store-sales-time-series-forecasting/oil.csv')
sample_submission = pd.read_csv('/kaggle/input/store-sales-time-series-forecasting/sample_submission.csv')
stores = pd.read_csv('/kaggle/input/store-sales-time-series-forecasting/stores.csv')
test = pd.read_csv('/kaggle/input/store-sales-time-series-forecasting/test.csv')
train = pd.read_csv('/kaggle/input/store-sales-time-series-forecasting/train.csv')
transactions = pd.read_csv('/kaggle/input/store-sales-time-series-forecasting/transactions.csv')

In [ ]:
data = (
    train
    .merge(stores, on='store_nbr', how='left')
    .merge(holidays_events, on='date', how='left')
    .merge(oil, on='date', how='left')
    .merge(transactions, on=['date', 'store_nbr'], how='left')
)

data.shape

In [ ]:
new_data = data.drop(['id'], axis=1)
new_data.shape

In [ ]:
new_data['store_nbr'] = new_data['store_nbr'].astype('category')
new_data['date'] = pd.to_datetime(new_data['date'])
new_data['family'] = new_data['family'].astype('category')
new_data['city'] = new_data['city'].astype('category')
new_data['state'] = new_data['state'].astype('category')
new_data['type_x'] = new_data['type_x'].astype('category') 
new_data['cluster'] = new_data['cluster'].astype('category') 

In [ ]:
new_data.info(show_counts=True)

---

## **Profiling**

In [ ]:
profile = ProfileReport(new_data, title="Profiling Report", explorative=True)
profile

**Итог:**

* Удалю колонки - type_y, locale, locale_name, description, transferred
* Удалю выбросы в колонках - sales, onpromotion
* Обработать пропуски - dcoilwtico, transactions
* Дубликатов - 1717

## **EDA**

### **Violinplot**

In [ ]:
plt.figure(figsize=(17, 5))
sns.violinplot(x='type_x', y='sales', data=new_data)
plt.title('Зависимость type от sale')
plt.show()

In [ ]:
plt.figure(figsize=(17, 5))
sns.violinplot(x='cluster', y='sales', data=new_data)
plt.title('Зависимость cluster от sale')
plt.show()

### **Scatterplot**

In [ ]:
plt.figure(figsize=(17, 5))
sns.scatterplot(x='onpromotion', y='sales', data=new_data)
plt.title('Зависимость onpromotion от sale')
plt.show()

In [ ]:
plt.figure(figsize=(17, 5))
sns.scatterplot(x='dcoilwtico', y='sales', data=new_data)
plt.title('Зависимость dcoilwtico от sale')
plt.show()

In [ ]:
plt.figure(figsize=(17, 5))
sns.scatterplot(x='transactions', y='sales', data=new_data)
plt.title('Зависимость transactions от sale')
plt.show()

---

## **Preproccesing**

### **FeatureExtraction**

In [ ]:
new_data = new_data.sort_values(['store_nbr', 'family', 'date'])
new_data

In [ ]:
new_data['lag_1'] = new_data.groupby(['store_nbr', 'family'])['sales'].shift(1)
new_data['lag_7'] = new_data.groupby(['store_nbr', 'family'])['sales'].shift(7)

new_data = new_data.dropna(subset=['lag_1', 'lag_7']).reset_index(drop=True)
new_data.shape

In [ ]:
for window in [3, 7, 14, 30]:
    new_data[f'rolling_mean_{window}'] = (
        new_data.groupby(['store_nbr', 'family'])['sales']
        .transform(lambda x: x.shift(1).rolling(window).mean())
    )
    new_data[f'rolling_std_{window}'] = (
        new_data.groupby(['store_nbr', 'family'])['sales']
        .transform(lambda x: x.shift(1).rolling(window).std())
    )

new_data = new_data.dropna(subset=[f'rolling_mean_{window}' for window in [3, 7, 14, 30]]).reset_index(drop=True)
new_data = new_data.dropna(subset=[f'rolling_std_{window}' for window in [3, 7, 14, 30]]).reset_index(drop=True)
new_data.shape

In [ ]:
new_data['year'] = new_data['date'].dt.year
new_data['month'] = new_data['date'].dt.month
new_data['day'] = new_data['date'].dt.day
new_data['day_of_week'] = new_data['date'].dt.dayofweek
new_data['is_weekend'] = new_data['day_of_week'].isin([5, 6]).astype(int)
new_data['weekofyear'] = new_data['date'].dt.isocalendar().week.astype(int)

new_data['month'] = new_data['month'].astype('category')
new_data['day_of_week'] = new_data['day_of_week'].astype('category')
new_data['is_weekend'] = new_data['is_weekend'].astype('category')

In [ ]:
new_data.info(show_counts=True)

### **Remove columns**

In [ ]:
new_data = new_data.drop(['type_y', 'locale', 'locale_name', 'description', 'transferred', 'dcoilwtico', 'transactions'], axis=1)
new_data.shape

### **Columns name**

In [ ]:
num_columns, cat_columns = new_data.select_dtypes(include=['number']).columns, new_data.select_dtypes(include=['object', 'category']).columns
num_columns.shape, cat_columns.shape

In [ ]:
num_columns

In [ ]:
cat_columns

### **Remove duplicates**

In [ ]:
new_data.drop_duplicates(inplace=True)
new_data.reset_index(drop=True, inplace=True)
new_data.shape

### **Outlier removal**

In [ ]:
no_nan_mask = new_data[num_columns].notna().all(axis=1)
clean_data = new_data[num_columns][no_nan_mask]


iso_forest = IsolationForest(contamination='auto', random_state=42)
iso_forest.fit(clean_data)
outliers = iso_forest.predict(clean_data)


clean_index = clean_data[outliers != -1].index
new_data_filter = new_data.loc[clean_index]

new_data = pd.concat([new_data_filter, new_data[~no_nan_mask]], ignore_index=True)
new_data.reset_index(drop=True, inplace=True)
new_data.shape

### **Selecte**

In [ ]:
new_data

In [ ]:
split_date = '2016-07-01'
before_2017 = new_data[new_data['date'] < split_date]
after_2017 = new_data[new_data['date'] >= split_date]

X_train = before_2017.drop(['sales', 'date'], axis=1)
y_train = before_2017['sales']
X_test = after_2017.drop(['sales', 'date'], axis=1)
y_test = after_2017['sales']

X_train.shape, X_test.shape, y_train.shape, y_test.shape

### **Missing value processing**

In [ ]:
plt.figure(figsize=(17, 5))
sns.heatmap(new_data.isna());

#### **Example 1**

In [ ]:
# num_transformer = Pipeline(steps=[
#     ('scaler', StandardScaler()),
#     ('imputer', KNNImputer(n_neighbors=5))
# ])

# preprocessor = ColumnTransformer(
#     transformers=[
#         ('num', num_transformer, num_columns),
#         ('cat', 'passthrough', cat_columns)
#     ],
#     n_jobs=-1
# )

# pipeline = Pipeline(steps=[
#     ('preprocessor', preprocessor)
# ])

# X_train = pipeline.fit_transform(X_train)
# X_test = pipeline.transform(X_test)

# fig, axe = plt.subplots(1, 2, figsize=(17, 5))
# sns.heatmap(X_train.isna(), ax=axe[0])
# sns.heatmap(X_test.isna(), ax=axe[1]);

### **Example 2**

In [ ]:
# num_transformer = Pipeline(steps=[
#     ('scaler', StandardScaler()),
#     ('imputer', SimpleImputer(strategy='mean'))
# ])

# preprocessor = ColumnTransformer(
#     transformers=[
#         ('num', num_transformer, num_columns.drop('sales')),
#         ('cat', 'passthrough', cat_columns)
#     ],
#     n_jobs=-1
# )

# pipeline = Pipeline(steps=[
#     ('preprocessor', preprocessor)
# ])

# X_train = pipeline.fit_transform(X_train)
# X_test = pipeline.transform(X_test)

# fig, axe = plt.subplots(1, 2, figsize=(17, 5))
# sns.heatmap(pd.isna(X_train), ax=axe[0])
# sns.heatmap(pd.isna(X_test), ax=axe[1]);

In [ ]:
# feature_names = preprocessor.get_feature_names_out()
# X_train = pd.DataFrame(X_train, columns=feature_names)
# X_test = pd.DataFrame(X_test, columns=feature_names)

# mask_duplicat = X_train.duplicated()
# X_train, y_train = X_train[~mask_duplicat].reset_index(drop=True), y_train.reset_index(drop=True)[~mask_duplicat].reset_index(drop=True)
# X_train.shape, y_train.shape

---

## **Models**

### **Pycaret**

In [ ]:
s = setup(
    data=new_data,
    target='sales',
    session_id=42,
    fold_strategy='timeseries',
    data_split_shuffle=False,
    fold_shuffle=False,
    normalize=True,
    transform_target=True,
    use_gpu=False,
    verbose=True
)

In [ ]:
my_models = ['lightgbm', 'xgboost', 'catboost']

In [ ]:
best = compare_models(include=my_models, fold=3)

In [ ]:
plot_model(best, plot='error')

In [ ]:
plot_model(best, plot='feature')

In [ ]:
plot_model(best, plot='residuals')

### **CatBoost Regressor**

In [ ]:
def root_mean_squared_log_error(y_true, y_pred):
    """Вычисляем RMSLE, избегая отрицательных значений"""
    y_true = np.clip(y_true, 0, None)
    y_pred = np.clip(y_pred, 0, None)
    return np.sqrt(mean_squared_log_error(y_true, y_pred))

In [ ]:
# def objective(trial):
#     start_time = time.time()
#     trial_number = trial.number
#     print(f"Запуск trial {trial_number}")

#     params = {
#         'iterations': trial.suggest_int('iterations', 300, 800),
#         'depth': trial.suggest_int('depth', 5, 9),
#         'learning_rate': trial.suggest_float('learning_rate', 0.02, 0.1, log=True),
#         'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1, 10),
#         'random_strength': trial.suggest_float('random_strength', 0.5, 1.5),
#         'min_data_in_leaf': trial.suggest_int('min_data_in_leaf', 10, 30),
#         'boosting_type': 'Plain',
#         'bootstrap_type': 'Bayesian',
#         'bagging_temperature': trial.suggest_float("bagging_temperature", 0.5, 1.5),
#         'task_type': 'CPU',
#         'loss_function': 'RMSE',
#         'thread_count': -1,
#         'random_seed': 42,
#         'verbose': False,
#         'early_stopping_rounds': 50
#     }

#     y_train_log = np.log1p(y_train)
#     split_ratio = 0.8
#     split_index = int(len(X_train) * split_ratio)

#     X_tr = X_train.iloc[:split_index]
#     X_val = X_train.iloc[split_index:]
#     y_tr = y_train_log.iloc[:split_index]
#     y_val = y_train_log.iloc[split_index:]

#     train_pool = Pool(X_tr, y_tr, cat_features=cat_columns.tolist())
#     val_pool = Pool(X_val, y_val, cat_features=cat_columns.tolist())

#     model = CatBoostRegressor(**params)
#     model.fit(train_pool, eval_set=val_pool, verbose=False)

#     preds_val = np.expm1(model.predict(val_pool))
#     rmsle = root_mean_squared_log_error(np.expm1(y_val), preds_val)

#     elapsed_time = time.time() - start_time
#     print(f"Trial {trial_number} завершён за {elapsed_time:.1f}с | RMSLE: {rmsle:.5f}")

#     return rmsle

In [ ]:
# study = optuna.create_study(direction='minimize')
# study.optimize(objective, n_trials=10)

# print("Лучшие гиперпараметры:", study.best_params)
# print("Лучшее значение RMSLE:", study.best_value)

In [ ]:
# best_params = study.best_params
# best_params.update({
#     "task_type": "CPU",
#     "loss_function": "RMSE",
#     "thread_count": -1,
#     "verbose": False,
#     "random_seed": 42
# })

# y_train_log = np.log1p(y_train)
# train_pool = Pool(X_train, y_train_log, cat_features=cat_columns.to_list())

# final_model = CatBoostRegressor(**best_params)
# final_model.fit(train_pool, early_stopping_rounds=100, verbose=False)

# preds = np.expm1(final_model.predict(X_test))
# rmsle = root_mean_squared_log_error(y_test, preds)
# print(f"Финальный RMSLE на тесте: {rmsle:.5f}")

### **Light Gradient Boosting Machine**

In [ ]:
def objective(trial):
    start_time = time.time()
    trial_number = trial.number
    print(f"Запуск trial {trial_number}")

    params = {
        'objective': 'regression',
        'metric': 'rmse',
        'verbosity': -1,
        'boosting_type': 'gbdt',
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'num_leaves': trial.suggest_int('num_leaves', 31, 255),
        'max_depth': trial.suggest_int('max_depth', 5, 12),
        'min_data_in_leaf': trial.suggest_int('min_data_in_leaf', 20, 100),
        'feature_fraction': trial.suggest_float('feature_fraction', 0.7, 1.0),
        'bagging_fraction': trial.suggest_float('bagging_fraction', 0.7, 1.0),
        'bagging_freq': trial.suggest_int('bagging_freq', 1, 5),
        'lambda_l1': trial.suggest_float('lambda_l1', 0.0, 5.0),
        'lambda_l2': trial.suggest_float('lambda_l2', 0.0, 5.0),
        'seed': 42
    }

    split_index = int(len(X_train) * 0.8)
    X_tr, X_val = X_train.iloc[:split_index], X_train.iloc[split_index:]
    y_tr, y_val = y_train.iloc[:split_index], y_train.iloc[split_index:]

    train_data = lgb.Dataset(X_tr, label=np.log1p(y_tr), categorical_feature=cat_columns.to_list())
    val_data = lgb.Dataset(X_val, label=np.log1p(y_val), categorical_feature=cat_columns.to_list())

    model = lgb.train(
        params,
        train_data,
        valid_sets=[val_data],
        num_boost_round=1000,
    )

    preds_val = np.expm1(model.predict(X_val))
    rmsle = root_mean_squared_log_error(y_val, preds_val)

    elapsed_time = time.time() - start_time
    print(f"Trial {trial_number} завершён за {elapsed_time:.1f}с | RMSLE: {rmsle:.5f}")

    return rmsle

In [ ]:
study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=10)

print("Лучшие гиперпараметры:", study.best_params)
print("Лучшее значение RMSLE:", study.best_value)

In [ ]:
best_params = study.best_params.copy()
best_params.update({
    "objective": "regression",
    "metric": "rmse",
    "verbosity": -1,
    "boosting_type": "gbdt",
    "seed": 42
})

y_train_log = np.log1p(y_train)
train_data = lgb.Dataset(X_train, label=y_train_log, categorical_feature=cat_columns.to_list())
val_data = lgb.Dataset(X_test, label=np.log1p(y_test), categorical_feature=cat_columns.to_list(), reference=train_data)

final_model = lgb.train(
    best_params,
    train_data,
    valid_sets=[val_data],
    num_boost_round=1000
)

preds = np.expm1(final_model.predict(X_test, num_iteration=final_model.best_iteration))
rmsle = root_mean_squared_log_error(y_test, preds)
print(f"Финальный RMSLE на тесте: {rmsle:.5f}")

In [ ]:
lgb.plot_importance(final_model, max_num_features=20)
plt.show()